# 🔍 03 — Drishti: SHAP Explainability

Generates per-prediction feature importance explanations. In the demo, judges see a waterfall plot:
- **Weather (PM2.5)** contributed +12 min
- **Previous station delay** contributed +8 min
- **Peak hour** contributed +3 min

This turns a black-box number into a story. Essential for **Accuracy/Effectiveness 25%** score.

In [0]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
# NO %pip install — shap, xgboost, matplotlib are pre-installed on DBR 14+.
# Installing them causes EnvironmentError because Databricks Serverless locks
# core packages (numpy, IPython) and blocks ABI-breaking version changes.

import json
import os
import pickle

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')   # non-interactive backend — required on Serverless
import matplotlib.pyplot as plt

import shap
import mlflow
import mlflow.xgboost

mlflow.set_registry_uri('databricks-uc')

VOLUME         = '/Volumes/setu_rail/bronze/raw_files'
MODEL_PKL_PATH = f'{VOLUME}/xgb_delay_model.pkl'
FEATURES_PATH  = f'{VOLUME}/xgb_feature_cols.json'
MODEL_UC_URI   = 'models:/setu_rail.gold.setu_delay_predictor@production'

print('✅ Imports OK')
print(f'   numpy   : {np.__version__}')
print(f'   shap    : {shap.__version__}')
print(f'   mlflow  : {mlflow.__version__}')

In [0]:
# ── Cell 2: Load XGBoost model + feature names ──────────────────────────────
# Variable name is xgb_model throughout this notebook — consistent.

# --- Feature names ---
if os.path.exists(FEATURES_PATH):
    with open(FEATURES_PATH, 'r') as f:
        feature_names = json.load(f)
    print(f'✅ Feature names loaded from Volume ({len(feature_names)} features)')
else:
    feature_names = [
        'stop_seq', 'total_stops', 'cumulative_travel_min', 'dwell_min',
        'scheduled_hour', 'is_peak_hour', 'pm25', 'no2', 'journey_day',
        'total_distance_km', 'duration_hours', 'latitude', 'longitude',
        'train_no_freq', 'station_code_freq'
    ]
    print(f'⚠️  Features JSON not found — using default list ({len(feature_names)} features)')
    print(f'   Tip: re-run notebook 02 to regenerate {FEATURES_PATH}')

print(f'Features: {feature_names}')

# --- XGBoost model (always stored as xgb_model) ---
if os.path.exists(MODEL_PKL_PATH):
    with open(MODEL_PKL_PATH, 'rb') as f:
        xgb_model = pickle.load(f)
    print(f'\n✅ Model loaded from Volume pickle: {MODEL_PKL_PATH}')
else:
    print(f'⚠️  Pickle not found at {MODEL_PKL_PATH} — loading from MLflow registry...')
    xgb_model = mlflow.xgboost.load_model(MODEL_UC_URI)
    print(f'✅ Model loaded from MLflow: {MODEL_UC_URI}')

print(f'Model type     : {type(xgb_model).__name__}')
print(f'n_estimators   : {xgb_model.n_estimators}')

In [0]:
# ── Cell 3: Load evaluation data from Delta (Spark) ─────────────────────────
# Read from the Gold Delta table — demonstrates Databricks Lakehouse depth.
# Reproduces the EXACT same encoding pipeline used during training in notebook 02.

from pyspark.sql.functions import col, count, coalesce, lit

df_spark = spark.table('setu_rail.gold.features_delay_ml')
print(f'Total rows in gold.features_delay_ml: {df_spark.count():,}')

# --- Frequency encode (must match notebook 02 exactly) ---
df_enc = df_spark.filter(col('arrival_delay_min').isNotNull())

for c in ['train_no', 'station_code']:
    if c in df_enc.columns:
        freq_map = (
            df_enc.groupBy(c)
                  .agg(count('*').alias('__cnt'))
                  .withColumnRenamed(c, f'__{c}')
        )
        df_enc = (
            df_enc
            .join(freq_map, df_enc[c] == freq_map[f'__{c}'], 'left')
            .withColumn(f'{c}_freq', coalesce(col('__cnt'), lit(1)).cast('double'))
            .drop('__cnt', f'__{c}')
        )
        print(f'  ✅ Frequency-encoded: {c}')

# --- Null fills (must match notebook 02 exactly) ---
fill_vals = {
    'stop_seq': 0, 'total_stops': 50, 'cumulative_travel_min': 0,
    'dwell_min': 5, 'scheduled_hour': 12, 'is_peak_hour': 0,
    'pm25': 60.0, 'no2': 25.0, 'journey_day': 1,
    'total_distance_km': 500, 'duration_hours': 8,
    'latitude': 20.0, 'longitude': 77.0,
    'train_no_freq': 1.0, 'station_code_freq': 1.0,
}
df_enc = df_enc.fillna({k: v for k, v in fill_vals.items() if k in df_enc.columns})

# --- Resolve which features actually exist in the table ---
available_features = [f for f in feature_names if f in df_enc.columns]
missing_features   = [f for f in feature_names if f not in df_enc.columns]

if missing_features:
    print(f'\n⚠️  Missing columns (will be zero-filled): {missing_features}')
    for mf in missing_features:
        df_enc = df_enc.withColumn(mf, lit(0.0))
    available_features = feature_names

# --- Collect 1000-row sample to Pandas ---
sample_pdf = (
    df_enc
    .select(available_features + ['arrival_delay_min'])
    .sample(fraction=0.005, seed=42)
    .limit(1000)
    .toPandas()
)

# Label-encode any string columns that slipped through (train_type, zone)
for c in list(sample_pdf.select_dtypes(include='object').columns):
    if c != 'arrival_delay_min':
        sample_pdf[c] = pd.Categorical(sample_pdf[c]).codes.astype(float)
        print(f'  ✅ Label-encoded: {c}')

X_sample = sample_pdf[available_features].values.astype(np.float32)
y_sample = sample_pdf['arrival_delay_min'].values

print(f'\n✅ Sample ready — X shape: {X_sample.shape}')
print(f'   Features: {available_features}')

In [0]:
# ── Cell 4: Compute SHAP values ─────────────────────────────────────────────
# Fixed: Use KernelExplainer (model-agnostic) instead of TreeExplainer
# TreeExplainer fails with corrupted XGBoost model metadata

print('Initialising SHAP KernelExplainer...')

# Use background-based KernelExplainer (more robust than TreeExplainer)
# Select 50 background samples for efficiency
background_data = shap.sample(X_sample, 50)

try:
    explainer = shap.KernelExplainer(
        model=xgb_model.predict,
        data=background_data,
        feature_names=available_features
    )
except Exception as e:
    print(f'⚠️  KernelExplainer failed: {e}')
    print('   Falling back to permutation-based Explainer...')
    # Ultimate fallback: permutation explainer (slowest but most robust)
    explainer = shap.Explainer(
        model=xgb_model.predict,
        data=background_data,
        feature_names=available_features
    )

n_explain  = min(500, len(X_sample))
X_explain  = X_sample[:n_explain]

print(f'Computing SHAP values for {n_explain} samples...')
shap_vals      = explainer.shap_values(X_explain)
expected_value = float(np.mean(xgb_model.predict(X_sample)))

print(f'\n✅ SHAP values computed')
print(f'   shap_vals shape : {shap_vals.shape}')
print(f'   Expected value  : {expected_value:.2f} min  (mean prediction across training data)')

# Persist to Volume so app.py can load them for the live demo
np.save(f'{VOLUME}/shap_values.npy', shap_vals)
np.save(f'{VOLUME}/shap_X.npy',      X_explain)

with open(f'{VOLUME}/shap_feature_names.json', 'w') as f:
    json.dump(available_features, f)

with open(f'{VOLUME}/shap_expected_value.json', 'w') as f:
    json.dump({'expected_value': expected_value}, f)

print(f'✅ Artifacts saved to {VOLUME}/')


In [0]:
# ── Cell 5: Global feature importance bar chart ──────────────────────────────

mean_abs_shap = np.abs(shap_vals).mean(axis=0)
order         = np.argsort(mean_abs_shap)   # ascending for horizontal bar

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    [available_features[i] for i in order],
    [mean_abs_shap[i]       for i in order],
    color='#e05c00', edgecolor='#b34500', linewidth=0.5,
)
ax.set_xlabel('Mean |SHAP value|  (minutes of delay attributed)', fontsize=11)
ax.set_title(
    'Drishti — Global Feature Importance\n(What drives Indian Railway delays?)',
    fontsize=12, fontweight='bold'
)
ax.tick_params(axis='y', labelsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

save_path_bar = f'{VOLUME}/shap_global_importance.png'
plt.savefig(save_path_bar, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {save_path_bar}')

print('\n📊 Feature importance ranking (top → bottom):')
for rank, i in enumerate(order[::-1], 1):
    bar = '█' * max(1, int(mean_abs_shap[i] / mean_abs_shap.max() * 25))
    print(f'  {rank:2d}. {available_features[i]:<30} {mean_abs_shap[i]:6.3f} min  {bar}')

In [0]:
# ── Cell 6: Waterfall plot — single high-delay prediction ───────────────────
# FIX: was 'model.predict(X_explain)' — correct variable is 'xgb_model'

y_pred_sample = xgb_model.predict(X_explain)
demo_idx      = int(np.argmax(y_pred_sample))
demo_pred     = float(y_pred_sample[demo_idx])
demo_actual   = float(y_sample[demo_idx])

print(f'Demo sample index : {demo_idx}')
print(f'Predicted delay   : {demo_pred:.1f} min')
print(f'Actual delay      : {demo_actual:.1f} min')

explanation = shap.Explanation(
    values        = shap_vals[demo_idx],
    base_values   = expected_value,
    data          = X_explain[demo_idx],
    feature_names = available_features,
)

plt.figure(figsize=(10, 5))
shap.waterfall_plot(explanation, max_display=12, show=False)
plt.title(f'Why was this train predicted {demo_pred:.0f} min late?', fontsize=11, pad=15)
plt.tight_layout()

save_path_wf = f'{VOLUME}/shap_waterfall_demo.png'
plt.savefig(save_path_wf, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {save_path_wf}')

In [0]:
# ── Cell 7: SHAP beeswarm summary plot ──────────────────────────────────────
# Shows DIRECTION of effect: high PM2.5 → positive SHAP (more delay).

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_vals,
    X_explain,
    feature_names = available_features,
    max_display   = 12,
    show          = False,
    plot_type     = 'dot',
)
plt.title('Drishti — SHAP Summary (Feature Direction & Magnitude)', fontsize=11)
plt.tight_layout()

save_path_summary = f'{VOLUME}/shap_summary_plot.png'
plt.savefig(save_path_summary, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {save_path_summary}')

In [0]:
# ── Cell 8: Log SHAP artifacts to MLflow ────────────────────────────────────

mlflow.set_experiment('/Shared/setu-rail-drishti')
top_feat_idx = int(np.argmax(mean_abs_shap))

with mlflow.start_run(run_name='shap_explainability_v1') as run:
    mlflow.log_artifact(save_path_bar,     'shap_plots')
    mlflow.log_artifact(save_path_wf,      'shap_plots')
    mlflow.log_artifact(save_path_summary, 'shap_plots')
    mlflow.log_artifact(f'{VOLUME}/shap_values.npy',          'shap_data')
    mlflow.log_artifact(f'{VOLUME}/shap_feature_names.json',  'shap_data')
    mlflow.log_artifact(f'{VOLUME}/shap_expected_value.json', 'shap_data')
    mlflow.log_metrics({
        'shap_expected_value':         expected_value,
        'shap_top_feature_importance': float(mean_abs_shap[top_feat_idx]),
        'num_features_explained':      float(len(available_features)),
        'num_samples_explained':       float(n_explain),
    })
    mlflow.log_param('top_feature', available_features[top_feat_idx])
    print(f'✅ MLflow run: {run.info.run_id}')

print('\n' + '=' * 60)
print('✅ SHAP EXPLAINABILITY COMPLETE')
print(f'   Top feature     : {available_features[top_feat_idx]}')
print(f'   Expected value  : {expected_value:.2f} min')
print(f'   Plots saved to  : {VOLUME}/')
print('=' * 60)
print('\n✅ Next: 04_cascade_simulator.ipynb')

✅ **Next:** `04_cascade_simulator.ipynb`